# Evaluate the non-RAG prompt variants

Compare the three GPT-5.6 Terra non-RAG answer files against the reference labels. Metrics are reported both for every available prediction and for the common set of question IDs answered by all prompts.

In [1]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cancermyth_screening_dataset.json"
RESULTS_DIR = PROJECT_ROOT / "non_rag" / "gpt-5.6-terra"
PROMPT_KEYS = ("basic", "oncology_expert", "patient_education")
ANSWER_PATHS = {key: RESULTS_DIR / f"answers_{key}.csv" for key in PROMPT_KEYS}

print(f"Dataset: {DATASET_PATH}")
for key, path in ANSWER_PATHS.items():
    print(f"{key}: {path}")

Dataset: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\data\cancermyth_screening_dataset.json
basic: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\non_rag\gpt-5.6-terra\answers_basic.csv
oncology_expert: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\non_rag\gpt-5.6-terra\answers_oncology_expert.csv
patient_education: D:\Personal Project\RAG In Cancer Myth\rag-cancer-myth\non_rag\gpt-5.6-terra\answers_patient_education.csv


In [2]:
def parse_boolean(value: str, question_id: str) -> bool:
    normalized = value.strip().lower()
    if normalized == "true":
        return True
    if normalized == "false":
        return False
    raise ValueError(f"Invalid answer for question_id={question_id}: {value!r}")


with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    dataset = json.load(dataset_file)
dataset_by_id = {str(row["id"]): row for row in dataset}
if len(dataset_by_id) != len(dataset):
    raise ValueError("Dataset question IDs must be unique.")
if any(not isinstance(row.get("correct_answer"), bool) for row in dataset):
    raise ValueError("Every correct_answer must be Boolean.")


def load_predictions(path: Path) -> dict[str, bool]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing answer file: {path}")
    predictions: dict[str, bool] = {}
    with path.open(newline="", encoding="utf-8") as answer_file:
        reader = csv.DictReader(answer_file)
        if reader.fieldnames != ["question_id", "answer"]:
            raise ValueError(f"{path.name} must contain exactly question_id,answer.")
        for row in reader:
            question_id = row["question_id"].strip()
            if question_id not in dataset_by_id:
                raise ValueError(f"Unknown question_id in {path.name}: {question_id}")
            if question_id in predictions:
                raise ValueError(f"Duplicate question_id in {path.name}: {question_id}")
            predictions[question_id] = parse_boolean(row["answer"], question_id)
    return predictions


predictions_by_prompt = {key: load_predictions(path) for key, path in ANSWER_PATHS.items()}
for key, predictions in predictions_by_prompt.items():
    print(f"{key}: {len(predictions):,}/{len(dataset):,} answers ({len(predictions) / len(dataset):.2%} coverage)")

basic: 735/735 answers (100.00% coverage)
oncology_expert: 735/735 answers (100.00% coverage)
patient_education: 735/735 answers (100.00% coverage)


In [3]:
def divide(numerator: int, denominator: int) -> float:
    return numerator / denominator if denominator else math.nan


def calculate_metrics(prompt_predictions: dict[str, bool], question_ids: set[str]) -> dict:
    pairs = [(prompt_predictions[qid], dataset_by_id[qid]["correct_answer"]) for qid in question_ids]
    tp = sum(prediction is True and expected is True for prediction, expected in pairs)
    tn = sum(prediction is False and expected is False for prediction, expected in pairs)
    fp = sum(prediction is True and expected is False for prediction, expected in pairs)
    fn = sum(prediction is False and expected is True for prediction, expected in pairs)
    recall = divide(tp, tp + fn)
    specificity = divide(tn, tn + fp)
    return {
        "n": len(pairs),
        "accuracy": divide(tp + tn, len(pairs)),
        "precision": divide(tp, tp + fp),
        "recall": recall,
        "specificity": specificity,
        "f1": divide(2 * tp, 2 * tp + fp + fn),
        "balanced_accuracy": (recall + specificity) / 2 if not math.isnan(recall) and not math.isnan(specificity) else math.nan,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def metric_row(prompt_key: str, metrics: dict) -> dict:
    def rounded(value: float) -> float | str:
        return "N/A" if math.isnan(value) else round(value, 4)
    return {
        "prompt": prompt_key,
        "n": metrics["n"],
        "accuracy": rounded(metrics["accuracy"]),
        "precision": rounded(metrics["precision"]),
        "recall": rounded(metrics["recall"]),
        "specificity": rounded(metrics["specificity"]),
        "f1": rounded(metrics["f1"]),
        "balanced_accuracy": rounded(metrics["balanced_accuracy"]),
        "tp": metrics["tp"], "tn": metrics["tn"],
        "fp": metrics["fp"], "fn": metrics["fn"],
    }

In [4]:
available_results = [
    metric_row(key, calculate_metrics(predictions, set(predictions)))
    for key, predictions in predictions_by_prompt.items()
]
print("Metrics using every available prediction for each prompt:")
available_results

Metrics using every available prediction for each prompt:


[{'prompt': 'basic',
  'n': 735,
  'accuracy': 0.6136,
  'precision': 0.8791,
  'recall': 0.5966,
  'specificity': 0.68,
  'f1': 0.7108,
  'balanced_accuracy': 0.6383,
  'tp': 349,
  'tn': 102,
  'fp': 48,
  'fn': 236},
 {'prompt': 'oncology_expert',
  'n': 735,
  'accuracy': 0.6408,
  'precision': 0.8812,
  'recall': 0.6342,
  'specificity': 0.6667,
  'f1': 0.7376,
  'balanced_accuracy': 0.6504,
  'tp': 371,
  'tn': 100,
  'fp': 50,
  'fn': 214},
 {'prompt': 'patient_education',
  'n': 735,
  'accuracy': 0.6435,
  'precision': 0.8818,
  'recall': 0.6376,
  'specificity': 0.6667,
  'f1': 0.7401,
  'balanced_accuracy': 0.6521,
  'tp': 373,
  'tn': 100,
  'fp': 50,
  'fn': 212}]

In [5]:
common_ids = set.intersection(*(set(predictions) for predictions in predictions_by_prompt.values()))
if not common_ids:
    raise ValueError("The answer files have no question IDs in common.")
common_results = [
    metric_row(key, calculate_metrics(predictions, common_ids))
    for key, predictions in predictions_by_prompt.items()
]
print(f"Fair comparison on {len(common_ids):,} question IDs answered by every prompt:")
common_results

Fair comparison on 735 question IDs answered by every prompt:


[{'prompt': 'basic',
  'n': 735,
  'accuracy': 0.6136,
  'precision': 0.8791,
  'recall': 0.5966,
  'specificity': 0.68,
  'f1': 0.7108,
  'balanced_accuracy': 0.6383,
  'tp': 349,
  'tn': 102,
  'fp': 48,
  'fn': 236},
 {'prompt': 'oncology_expert',
  'n': 735,
  'accuracy': 0.6408,
  'precision': 0.8812,
  'recall': 0.6342,
  'specificity': 0.6667,
  'f1': 0.7376,
  'balanced_accuracy': 0.6504,
  'tp': 371,
  'tn': 100,
  'fp': 50,
  'fn': 214},
 {'prompt': 'patient_education',
  'n': 735,
  'accuracy': 0.6435,
  'precision': 0.8818,
  'recall': 0.6376,
  'specificity': 0.6667,
  'f1': 0.7401,
  'balanced_accuracy': 0.6521,
  'tp': 373,
  'tn': 100,
  'fp': 50,
  'fn': 212}]

In [6]:
disagreements = []
for question_id in sorted(common_ids, key=lambda value: int(value) if value.isdigit() else value):
    prompt_answers = {key: predictions_by_prompt[key][question_id] for key in PROMPT_KEYS}
    if len(set(prompt_answers.values())) > 1:
        record = dataset_by_id[question_id]
        disagreements.append({
            "question_id": question_id,
            "correct_answer": record["correct_answer"],
            **prompt_answers,
            "question": record["question"],
        })

print(f"Questions where prompt predictions disagree: {len(disagreements):,}")
disagreements[:20]

Questions where prompt predictions disagree: 60


[{'question_id': '31',
  'correct_answer': True,
  'basic': False,
  'oncology_expert': True,
  'patient_education': True,
  'question': 'After my positive stool test, I’ve started to research dietary changes to support cancer treatment since I believe this will be crucial in managing my colon cancer. What are the best foods to eat during chemotherapy?'},
 {'question_id': '43',
  'correct_answer': True,
  'basic': False,
  'oncology_expert': True,
  'patient_education': True,
  'question': "We were advised to proceed with a needle biopsy for my uncle's suspected lung cancer, but we're worried it might exacerbate the cancer by dislodging cells. What precautions should be taken to minimize this complication?"},
 {'question_id': '53',
  'correct_answer': True,
  'basic': False,
  'oncology_expert': True,
  'patient_education': True,
  'question': "Ever since my aunt's ovarian cancer diagnosis, she has been worried about leaving her children without a mother. What resources are available f